# NB10 — Weight Randomization Sanity Check (Adebayo et al., 2018)

**Scientific Goal:**
Verify that Integrated Gradients (IG) attribution maps reflect the *model's learned
parameters* rather than structural input biases or architectural constants.

**Method (Adebayo et al., 2018 — "Sanity Checks for Saliency Maps"):**
A faithful saliency method must produce attribution maps that degrade to noise-like
outputs when the underlying model's weights are randomized. We perform two schemes:

1. **Full randomization**: ALL layers randomized simultaneously → maps should be
   completely unstructured (SSIM ≈ 0, Spearman ρ ≈ 0).
2. **Cascading (top-down)**: Layers randomized progressively from the classification
   head downward at 25 / 50 / 75 / 100 % checkpoints → SSIM should decrease
   monotonically, confirming sensitivity to each layer's contribution.

**Four Divergence Metrics:**
- SSIM (Structural Similarity Index)
- Spearman rank correlation (ρ)
- KL Divergence of attribution mass histograms
- Entropy ratio (H_randomized / H_trained)

**Inputs required** (all produced by NB04):
  results/ig_manifest.csv, ig_maps/*.npy, models/*_finetuned.pt

**GPU required** for IG computation on randomized models.
Baseline IG maps are loaded from disk (no GPU needed for that step).

In [1]:
from google.colab import drive
drive.mount('/content/drive')

GDRIVE_ROOT = '/content/drive/MyDrive/cxr_faithfulness'
exec(open(f'{GDRIVE_ROOT}/config/startup.py').read())

Mounted at /content/drive
⏳ Installing strictly pinned architecture packages onto Colab's native stack (~30s)...
✅ Packages ready! Using native modern PyTorch and NumPy.


In [2]:
# Pinned installs — identical to NB04 / NB09 plus scikit-image for SSIM
import subprocess, sys
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'timm==0.9.12', 'captum==0.7.0', 'peft==0.6.2',
])

import torch
assert torch.cuda.is_available(), (
    "GPU required — switch to a GPU runtime in Colab (Runtime > Change runtime type > T4 GPU)"
)
print(f"✅ GPU: {torch.cuda.get_device_name(0)}")

✅ GPU: Tesla T4


In [3]:
import os, gc, copy, json, random, warnings
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import scipy.stats as stats
from skimage.metrics import structural_similarity as skimage_ssim
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.models as tvmodels
import timm
from torch.amp import autocast
from captum.attr import IntegratedGradients
from peft import LoraConfig, TaskType, get_peft_model

# ── Paths ──────────────────────────────────────────────────────────────────
ROOT          = Path(GDRIVE_ROOT)
IMAGES_PATH   = ROOT / 'data' / 'processed' / 'images'
SPLITS_PATH   = ROOT / 'data' / 'processed' / 'splits'
MODELS_PATH   = ROOT / 'models'
RESULTS_PATH  = ROOT / 'results'
IG_PATH       = ROOT / 'ig_maps'
FIGPATH       = ROOT / 'figures'
RAND_PATH     = ROOT / 'ig_maps' / 'randomized'

for p in [RESULTS_PATH, FIGPATH, RAND_PATH]:
    p.mkdir(parents=True, exist_ok=True)

# ── Constants ───────────────────────────────────────────────────────────────
IMG_SIZE      = 224
NUM_CLASSES   = 14
RANDOM_SEED   = 42
IG_STEPS      = 300          # match NB04
DELTA_THRESH  = 0.01         # convergence threshold
N_SAMPLE      = 30           # stratified images per model
MODEL_NAMES   = ['densenet121', 'convnextv2_tiny', 'swinb_lora']
CASCADE_FRACS = [0.25, 0.50, 0.75, 1.00]   # layer fraction checkpoints

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

print("✅ Imports and paths ready.")

✅ Imports and paths ready.


In [4]:
# ── Reproducibility & image helpers ────────────────────────────────────────

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


test_transforms = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


def load_image_tensor(image_id: str) -> torch.Tensor:
    img_p = IMAGES_PATH / f"{image_id}.png"
    img = cv2.imread(str(img_p))
    assert img is not None, f"Missing image: {img_p}"
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return test_transforms(img).unsqueeze(0).cuda()


def normalize_ig_map(attr: np.ndarray) -> np.ndarray:
    """Abs-sum over channels, min-max normalize to [0, 1]."""
    attr = np.abs(attr)
    if attr.ndim == 3:
        attr = attr.sum(axis=0)
    mn, mx = attr.min(), attr.max()
    if (mx - mn) < 1e-8:
        return np.zeros_like(attr, dtype=np.float32)
    return ((attr - mn) / (mx - mn)).astype(np.float32)


class IgCaptumWrapper(nn.Module):
    """PEFT + timm Swin: LoraModel.forward(x), never HF input_ids kwargs."""
    def __init__(self, model: nn.Module):
        super().__init__()
        self.model = model
    def forward(self, x: torch.Tensor, *args, **kwargs) -> torch.Tensor:
        kwargs.pop('input_ids', None)
        kwargs.pop('attention_mask', None)
        base = getattr(self.model, 'base_model', None)
        if base is not None:
            return base(x, *args, **kwargs)
        return self.model(x, *args, **kwargs)


def captum_model(model: nn.Module, model_name: str) -> nn.Module:
    return IgCaptumWrapper(model) if model_name == 'swinb_lora' else model


def run_ig(model: nn.Module, img_tensor: torch.Tensor,
           target_idx: int, n_steps: int, model_name: str):
    ig = IntegratedGradients(captum_model(model, model_name))
    baseline = torch.zeros_like(img_tensor)
    attr, delta = ig.attribute(
        img_tensor,
        baselines=baseline,
        target=target_idx,
        n_steps=n_steps,
        internal_batch_size=50,
        return_convergence_delta=True,
    )
    return attr.detach().cpu().numpy()[0], float(delta.detach().cpu().item())


def resolve_ig_path(path_str: str) -> Path:
    """Resolve Colab-absolute paths to local Drive-mounted paths."""
    p = Path(str(path_str).replace(
        '/content/drive/MyDrive/cxr_faithfulness', str(ROOT)
    ))
    if p.exists():
        return p
    # Fallback: strip to suffix after 'cxr_faithfulness/'
    parts = str(path_str).replace('\\', '/').split('cxr_faithfulness/')
    if len(parts) > 1:
        candidate = ROOT / parts[-1]
        if candidate.exists():
            return candidate
    return IG_PATH / Path(path_str).name


print("✅ Helpers defined.")

✅ Helpers defined.


In [5]:
# ── Model loader (identical to NB04 / NB09) ────────────────────────────────

sweep_df = pd.read_csv(MODELS_PATH / 'lora_sweep.csv')
selected_rank = int(sweep_df.loc[sweep_df['val_auc'].idxmax(), 'rank'])

with open(MODELS_PATH / 'thresholds.json') as f:
    all_thresholds = json.load(f)

_test = pd.read_csv(SPLITS_PATH / 'test_patho.csv', nrows=0)
LABEL_COLS = [c for c in _test.columns if c != 'image_id']
assert len(LABEL_COLS) == NUM_CLASSES, (
    f"Expected {NUM_CLASSES} classes, got {len(LABEL_COLS)}: {LABEL_COLS}"
)


def load_model(model_name: str) -> nn.Module:
  strict_mode = True
  if model_name == 'densenet121':
    m = tvmodels.densenet121(weights=None)
    m.classifier = nn.Linear(1024, NUM_CLASSES)
  elif model_name == 'convnextv2_tiny':
    m = timm.create_model('convnextv2_tiny.fcmae_ft_in22k_in1k', pretrained=False, num_classes=0)
    m.head.fc = nn.Linear(768, NUM_CLASSES)
  elif model_name == 'swinb_lora':
    base = timm.create_model('swin_base_patch4_window7_224', pretrained=False, num_classes=NUM_CLASSES)
    lora_cfg = LoraConfig(
      r=selected_rank, lora_alpha=selected_rank * 2,
      target_modules=['qkv', 'proj'], lora_dropout=0.1,
      bias='none',
    )
    m = get_peft_model(base, lora_cfg)
    strict_mode = False
  else:
    raise ValueError(f"Unknown model: {model_name}")

  ckpt_path = MODELS_PATH / f'{model_name}_finetuned.pt'
  assert ckpt_path.exists(), f"Missing checkpoint: {ckpt_path}"
  state = torch.load(str(ckpt_path), map_location='cpu', weights_only=True)
  res = m.load_state_dict(state, strict=strict_mode)

  if not strict_mode:
    ALLOWED = ('base_model.', 'lora_', 'base_layer.')
    bad = [k for k in res.missing_keys if not k.startswith(ALLOWED)]
    assert not bad, f"Backbone keys missing: {bad[:10]}"

  m = m.cuda().eval()
  print(f"  ✅ {model_name} loaded")
  return m


print("✅ Model loader defined.")

✅ Model loader defined.


In [6]:
# ── Randomization helpers ──────────────────────────────────────────────────

def reset_batchnorm_stats(model: nn.Module) -> None:
    """Reset BN running stats — required when conv weights are reinitialized."""
    for mod in model.modules():
        if isinstance(mod, nn.BatchNorm2d):
            mod.running_mean.zero_()
            mod.running_var.fill_(1.0)
            if mod.num_batches_tracked is not None:
                mod.num_batches_tracked.zero_()


def get_randomizable_layers(model: nn.Module) -> list:
  """
  All Linear/Conv/Norm layers with weights, top-down order.
  Includes FROZEN backbone layers (required for Swin Adebayo test).
  """
  forward_order = [
    (name, mod) for name, mod in model.named_modules()
    if isinstance(mod, (nn.Linear, nn.Conv2d,
                        nn.LayerNorm, nn.BatchNorm2d, nn.GroupNorm))
    and getattr(mod, 'weight', None) is not None
  ]
  return list(reversed(forward_order))


def randomize_layers(model: nn.Module,
                     layers_to_rand: list,
                     seed: int = 42) -> nn.Module:
    """
    Deep-copies the model and reinitializes only the specified layers.
    Does NOT mutate the original model.

    Initialization scheme:
      - Linear / Conv2d weights: Kaiming-normal (suitable for ReLU/GELU nets)
      - Linear / Conv2d biases:  zeros
      - LayerNorm / BN / GN weight (gamma): ones   (preserve scale)
      - LayerNorm / BN / GN bias  (beta):   zeros  (preserve shift)
      - BN running_mean / running_var: reset to 0 / 1 when BN or any Conv2d
        in this subset is reinitialized (trained stats + random conv → NaN IG)
    """
    m = copy.deepcopy(model)
    device = next(m.parameters()).device
    rng = torch.Generator(device=device)
    rng.manual_seed(seed)

    layer_names = {name for name, _ in layers_to_rand}
    touched_conv = False
    for name, mod in m.named_modules():
        if name not in layer_names:
            continue
        with torch.no_grad():
            if isinstance(mod, (nn.Linear, nn.Conv2d)):
                nn.init.kaiming_normal_(mod.weight,
                                        nonlinearity='relu',
                                        generator=rng)
                if mod.bias is not None:
                    nn.init.zeros_(mod.bias)
                if isinstance(mod, nn.Conv2d):
                    touched_conv = True
            elif isinstance(mod, (nn.LayerNorm, nn.BatchNorm2d, nn.GroupNorm)):
                if mod.weight is not None:
                    nn.init.ones_(mod.weight)
                if mod.bias is not None:
                    nn.init.zeros_(mod.bias)
                if isinstance(mod, nn.BatchNorm2d):
                    mod.running_mean.zero_()
                    mod.running_var.fill_(1.0)
                    if mod.num_batches_tracked is not None:
                        mod.num_batches_tracked.zero_()

    if touched_conv:
        reset_batchnorm_stats(m)

    return m.cuda().eval()


print("✅ Randomization helpers defined.")

✅ Randomization helpers defined.


In [7]:
# ── Stratified sample selection ────────────────────────────────────────────
set_seed(RANDOM_SEED)

manifest = pd.read_csv(RESULTS_PATH / 'ig_manifest.csv')
manifest['image_id'] = manifest['image_id'].astype(str)
manifest_patho = manifest[manifest['subset'] == 'patho'].copy()


def stratified_sample(df: pd.DataFrame, n: int, seed: int = RANDOM_SEED) -> pd.DataFrame:
    """
    Proportional stratified sample by target_class.
    Falls back gracefully when a class has fewer rows than the per-class quota.
    """
    rng = np.random.default_rng(seed)
    classes = df['target_class'].unique()
    per_class = max(1, n // len(classes))
    chunks = []
    for cls in classes:
        sub = df[df['target_class'] == cls]
        k = min(per_class, len(sub))
        idx = rng.choice(len(sub), size=k, replace=False)
        chunks.append(sub.iloc[idx])
    sample = pd.concat(chunks, ignore_index=True)
    if len(sample) > n:
        idx = rng.choice(len(sample), size=n, replace=False)
        sample = sample.iloc[idx]
    return sample.reset_index(drop=True)


samples = {}
for model_name in MODEL_NAMES:
    sub = manifest_patho[manifest_patho['model'] == model_name].copy()
    s = stratified_sample(sub, N_SAMPLE)
    samples[model_name] = s
    print(f"  {model_name}: {len(s)} samples, "
          f"{s['target_class'].nunique()} pathologies "
          f"({', '.join(sorted(s['target_class'].unique()))})")

print(f"\n✅ Sample selection complete — N={N_SAMPLE} per model.")

  densenet121: 29 samples, 10 pathologies (Aortic enlargement, Cardiomegaly, ILD, Infiltration, Lung Opacity, Nodule/Mass, Other lesion, Pleural effusion, Pleural thickening, Pulmonary fibrosis)
  convnextv2_tiny: 20 samples, 11 pathologies (Aortic enlargement, Calcification, Cardiomegaly, ILD, Infiltration, Lung Opacity, Nodule/Mass, Other lesion, Pleural effusion, Pleural thickening, Pulmonary fibrosis)
  swinb_lora: 27 samples, 9 pathologies (Aortic enlargement, Cardiomegaly, ILD, Infiltration, Lung Opacity, Nodule/Mass, Pleural effusion, Pleural thickening, Pulmonary fibrosis)

✅ Sample selection complete — N=30 per model.


In [8]:
# ── Load baseline IG maps from disk (CPU-only, no GPU needed) ──────────────

baseline_maps = {}   # (image_id, model_name) -> np.ndarray (224, 224) float32

for model_name in MODEL_NAMES:
    df = samples[model_name]
    loaded = 0
    for _, row in df.iterrows():
        ig_p = resolve_ig_path(row['ig_path'])
        if not ig_p.exists():
            warnings.warn(f"Missing baseline IG: {ig_p}")
            continue
        attr = np.load(str(ig_p))
        baseline_maps[(row['image_id'], model_name)] = normalize_ig_map(attr)
        loaded += 1
    print(f"  {model_name}: {loaded}/{len(df)} baseline maps loaded from disk")

assert len(baseline_maps) >= len(MODEL_NAMES) * 5, (
    f"Only {len(baseline_maps)} baseline maps found — check ig_maps/ paths. "
    "Ensure NB04 ran successfully."
)
print(f"\n✅ Total baseline maps loaded: {len(baseline_maps)}")

  densenet121: 29/29 baseline maps loaded from disk
  convnextv2_tiny: 20/20 baseline maps loaded from disk
  swinb_lora: 27/27 baseline maps loaded from disk

✅ Total baseline maps loaded: 76


In [9]:
# ── Divergence metric helpers ──────────────────────────────────────────────

def compute_ssim(a: np.ndarray, b: np.ndarray) -> float:
    """SSIM between two (H, W) float maps in [0, 1]."""
    a = np.nan_to_num(a, nan=0.0)
    b = np.nan_to_num(b, nan=0.0)
    if a.size == 0 or b.size == 0:
        return 0.0
    val = skimage_ssim(a, b, data_range=1.0)
    return 0.0 if not np.isfinite(val) else float(val)


def compute_spearman(a: np.ndarray, b: np.ndarray) -> float:
    """Spearman rank correlation between flattened maps."""
    a = np.nan_to_num(a.ravel(), nan=0.0)
    b = np.nan_to_num(b.ravel(), nan=0.0)
    if np.std(a) < 1e-8 or np.std(b) < 1e-8:
        return 0.0
    r, _ = stats.spearmanr(a, b)
    return 0.0 if not np.isfinite(r) else float(r)


def compute_kld(a: np.ndarray, b: np.ndarray, n_bins: int = 50,
                eps: float = 1e-10) -> float:
    """
    KL divergence D_KL(trained ‖ randomized), computed via 50-bin histogram
    over [0, 1]. Both histograms normalized to valid probability distributions.
    """
    p_hist, _ = np.histogram(a, bins=n_bins, range=(0.0, 1.0), density=False)
    q_hist, _ = np.histogram(b, bins=n_bins, range=(0.0, 1.0), density=False)
    p = p_hist.astype(np.float64) + eps
    q = q_hist.astype(np.float64) + eps
    p /= p.sum()
    q /= q.sum()
    val = float(np.sum(p * np.log(p / q)))
    return 0.0 if not np.isfinite(val) else val


def compute_entropy_ratio(trained: np.ndarray, rand: np.ndarray,
                          eps: float = 1e-10) -> float:
    """
    Ratio H(rand) / H(trained) of spatial Shannon entropy.
    Values approaching or exceeding 1.0 indicate the randomized map
    is at least as entropic (diffuse) as the trained map — expected for noise.
    """
    def _entropy(m: np.ndarray) -> float:
        flat = m.ravel().astype(np.float64)
        total = flat.sum()
        if total < eps:
            return 0.0
        p = flat / total
        return float(-np.sum(p * np.log(p + eps)))

    H_trained = _entropy(trained)
    H_rand    = _entropy(rand)
    return float(H_rand / (H_trained + eps))


def compute_all_metrics(baseline: np.ndarray, randomized: np.ndarray) -> dict:
    return {
        'ssim':          compute_ssim(baseline, randomized),
        'spearman_r':    compute_spearman(baseline, randomized),
        'kld':           compute_kld(baseline, randomized),
        'entropy_ratio': compute_entropy_ratio(baseline, randomized),
    }


print("✅ Metric helpers defined.")

✅ Metric helpers defined.


In [10]:
# ── PASS 1: Full randomization (all layers simultaneously) ─────────────────

print("=" * 65)
print("PASS 1: Full Randomization (all layers simultaneously)")
print("=" * 65)

full_rand_rows = []
set_seed(RANDOM_SEED)

for model_name in MODEL_NAMES:
    print(f"\n--- {model_name} ---")
    trained_model = load_model(model_name)

    rand_layers = get_randomizable_layers(trained_model)
    print(f"  {model_name}: {len(rand_layers)} randomizable layers")

    rand_model = randomize_layers(trained_model, rand_layers, seed=RANDOM_SEED)
    df = samples[model_name]
    n_ok = 0

    for _, row in df.iterrows():
        key = (row['image_id'], model_name)
        if key not in baseline_maps:
            continue
        try:
            img_tensor = load_image_tensor(row['image_id'])
            target_idx = int(row['target_idx'])

            attr_rand, _ = run_ig(rand_model, img_tensor, target_idx, IG_STEPS, model_name)
            attr_rand_norm = normalize_ig_map(attr_rand)
            if not np.isfinite(attr_rand_norm).all():
                warnings.warn(f"  Skip {row['image_id']}/{model_name}: non-finite IG map")
                continue

            metrics = compute_all_metrics(baseline_maps[key], attr_rand_norm)
            metrics.update({
                'image_id':          row['image_id'],
                'model':             model_name,
                'target_class':      row['target_class'],
                'scheme':            'full',
                'n_layers_rand':     len(rand_layers),
                'pct_layers_rand':   1.0,
            })
            full_rand_rows.append(metrics)

            # Save randomized map for figure generation
            rand_p = RAND_PATH / f"{row['image_id']}_{model_name}_full_rand.npy"
            np.save(str(rand_p), attr_rand_norm)
            n_ok += 1

        except Exception as e:
            warnings.warn(f"  Skip {row['image_id']}/{model_name}: {e}")

    print(f"  Processed: {n_ok}/{len(df)}")
    del trained_model, rand_model
    gc.collect()
    torch.cuda.empty_cache()

full_rand_df = pd.DataFrame(full_rand_rows)
full_rand_df.to_csv(RESULTS_PATH / 'weight_rand_full.csv', index=False)
print(f"\n✅ Full randomization complete — {len(full_rand_df)} rows saved.")
print()

# Quick summary table
print(f"{'Model':<22} {'SSIM':>8} {'Spearman_r':>12} {'KLD':>8} {'Entr.ratio':>12}")
print("-" * 65)
for mn in MODEL_NAMES:
    sub = full_rand_df[full_rand_df['model'] == mn]
    if sub.empty:
        continue
    print(f"{mn:<22} {sub['ssim'].mean():>8.4f} {sub['spearman_r'].mean():>12.4f} "
          f"{sub['kld'].mean():>8.4f} {sub['entropy_ratio'].mean():>12.4f}")

PASS 1: Full Randomization (all layers simultaneously)

--- densenet121 ---
  ✅ densenet121 loaded
  densenet121: 242 randomizable layers
  Processed: 29/29

--- convnextv2_tiny ---
  ✅ convnextv2_tiny loaded
  convnextv2_tiny: 82 randomizable layers
  Processed: 20/20

--- swinb_lora ---


/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


  ✅ swinb_lora loaded
  swinb_lora: 252 randomizable layers
  Processed: 27/27

✅ Full randomization complete — 76 rows saved.

Model                      SSIM   Spearman_r      KLD   Entr.ratio
-----------------------------------------------------------------
densenet121              0.1597       0.5115   0.7427       1.0294
convnextv2_tiny          0.0770       0.2633   1.4903       1.0270
swinb_lora               0.2556      -0.1644   0.7708       0.9272


In [11]:
# ── PASS 2: Cascading randomization (top-down, 4 checkpoints) ─────────────

print("=" * 65)
print("PASS 2: Cascading Randomization (classifier head → input)")
print(f"Checkpoints: {[f'{int(f*100)}%' for f in CASCADE_FRACS]}")
print("=" * 65)

cascade_rows = []
set_seed(RANDOM_SEED)

for model_name in MODEL_NAMES:
    print(f"\n--- {model_name} ---")
    trained_model = load_model(model_name)
    rand_layers_ordered = get_randomizable_layers(trained_model)   # top-down
    n_total = len(rand_layers_ordered)
    print(f"  Total randomizable layers (top-down): {n_total}")

    # Build (frac, k) checkpoints, deduplicated
    checkpoints = sorted(
        {(frac, max(1, int(round(frac * n_total)))) for frac in CASCADE_FRACS},
        key=lambda x: x[1]
    )

    df = samples[model_name]

    for frac, k in checkpoints:
        layers_subset = rand_layers_ordered[:k]   # classifier head first
        rand_model = randomize_layers(trained_model, layers_subset,
                                      seed=RANDOM_SEED + k)
        n_ok = 0
        checkpoint_ssims = []

        for _, row in df.iterrows():
            key = (row['image_id'], model_name)
            if key not in baseline_maps:
                continue
            try:
                img_tensor = load_image_tensor(row['image_id'])
                target_idx = int(row['target_idx'])

                attr_rand, _ = run_ig(rand_model, img_tensor, target_idx, IG_STEPS, model_name)
                attr_rand_norm = normalize_ig_map(attr_rand)
                if not np.isfinite(attr_rand_norm).all():
                    warnings.warn(f"  Skip {row['image_id']}: non-finite IG map")
                    continue

                metrics = compute_all_metrics(baseline_maps[key], attr_rand_norm)
                metrics.update({
                    'image_id':        row['image_id'],
                    'model':           model_name,
                    'target_class':    row['target_class'],
                    'scheme':          'cascading',
                    'n_layers_rand':   k,
                    'pct_layers_rand': round(frac, 4),
                })
                cascade_rows.append(metrics)
                checkpoint_ssims.append(metrics['ssim'])
                n_ok += 1

            except Exception as e:
                warnings.warn(f"  Skip {row['image_id']}: {e}")

        mean_ssim = np.mean(checkpoint_ssims) if checkpoint_ssims else float('nan')
        print(f"  {frac*100:.0f}% ({k:3d} layers): SSIM={mean_ssim:.4f} | n={n_ok}")
        del rand_model
        gc.collect()
        torch.cuda.empty_cache()

    del trained_model
    gc.collect()
    torch.cuda.empty_cache()

cascade_df = pd.DataFrame(cascade_rows)
cascade_df.to_csv(RESULTS_PATH / 'weight_rand_cascade.csv', index=False)
print(f"\n✅ Cascading randomization complete — {len(cascade_df)} rows saved.")

PASS 2: Cascading Randomization (classifier head → input)
Checkpoints: ['25%', '50%', '75%', '100%']

--- densenet121 ---
  ✅ densenet121 loaded
  Total randomizable layers (top-down): 242
  25% ( 60 layers): SSIM=0.2053 | n=29
  50% (121 layers): SSIM=0.2269 | n=29
  75% (182 layers): SSIM=0.2363 | n=29
  100% (242 layers): SSIM=0.1706 | n=29

--- convnextv2_tiny ---
  ✅ convnextv2_tiny loaded
  Total randomizable layers (top-down): 82
  25% ( 20 layers): SSIM=0.4785 | n=20
  50% ( 41 layers): SSIM=0.4248 | n=20
  75% ( 62 layers): SSIM=0.3574 | n=20
  100% ( 82 layers): SSIM=0.0566 | n=20

--- swinb_lora ---
  ✅ swinb_lora loaded
  Total randomizable layers (top-down): 252
  25% ( 63 layers): SSIM=0.4578 | n=27
  50% (126 layers): SSIM=0.4284 | n=27
  75% (189 layers): SSIM=0.3412 | n=27
  100% (252 layers): SSIM=0.2350 | n=27

✅ Cascading randomization complete — 304 rows saved.


In [12]:
# ── Aggregate results table with bootstrap CIs ────────────────────────────

def boot_ci(values, n_boot: int = 1000, ci: float = 0.95,
            seed: int = RANDOM_SEED):
    vals = np.asarray(values, dtype=float)
    vals = vals[~np.isnan(vals)]
    if len(vals) == 0:
        return np.nan, np.nan
    rng = np.random.default_rng(seed)
    boots = np.array([
        rng.choice(vals, size=len(vals), replace=True).mean()
        for _ in range(n_boot)
    ])
    lo_q = (1 - ci) / 2
    return float(np.quantile(boots, lo_q)), float(np.quantile(boots, 1 - lo_q))


agg_rows = []
for model_name in MODEL_NAMES:
    row = {'model': model_name}

    # Full randomization metrics
    sub_full = full_rand_df[full_rand_df['model'] == model_name]
    for met in ['ssim', 'spearman_r', 'kld', 'entropy_ratio']:
        vals = sub_full[met].dropna().values
        lo, hi = boot_ci(vals)
        row[f'full_{met}_mean'] = round(float(vals.mean()), 4) if len(vals) else np.nan
        row[f'full_{met}_ci_lo'] = round(lo, 4)
        row[f'full_{met}_ci_hi'] = round(hi, 4)

    # Cascading SSIM at each checkpoint
    sub_cas = cascade_df[cascade_df['model'] == model_name]
    for frac in CASCADE_FRACS:
        sub_f = sub_cas[np.isclose(sub_cas['pct_layers_rand'], frac, atol=0.01)]
        vals = sub_f['ssim'].dropna().values
        lo, hi = boot_ci(vals)
        tag = f'cascade_ssim_pct{int(round(frac * 100))}'
        row[f'{tag}_mean'] = round(float(vals.mean()), 4) if len(vals) else np.nan
        row[f'{tag}_ci_lo'] = round(lo, 4)
        row[f'{tag}_ci_hi'] = round(hi, 4)

    agg_rows.append(row)

agg_df = pd.DataFrame(agg_rows)
agg_df.to_csv(RESULTS_PATH / 'weight_randomization_results.csv', index=False)
print("✅ weight_randomization_results.csv saved.")
print()
print(agg_df[['model',
              'full_ssim_mean', 'full_spearman_r_mean',
              'full_kld_mean',  'full_entropy_ratio_mean']].to_string(index=False))

✅ weight_randomization_results.csv saved.

          model  full_ssim_mean  full_spearman_r_mean  full_kld_mean  full_entropy_ratio_mean
    densenet121          0.1597                0.5115         0.7427                   1.0294
convnextv2_tiny          0.0770                0.2633         1.4903                   1.0270
     swinb_lora          0.2556               -0.1644         0.7708                   0.9272


In [13]:
# ── Figures ────────────────────────────────────────────────────────────────

print("\nGenerating figures...")

# ── Figure 1: Visual comparison panel ──────────────────────────────────────
# Trained IG | Fully Randomized IG | Absolute Difference
# One representative image per model (median SSIM row)

fig, axes = plt.subplots(len(MODEL_NAMES), 3,
                          figsize=(12, 4.2 * len(MODEL_NAMES)))
fig.suptitle(
    "Weight Randomization Sanity Check (Adebayo et al., 2018)\n"
    "Trained vs. Fully Randomized Model — Integrated Gradients",
    fontsize=13, fontweight='bold', y=1.01,
)

col_titles = ['Trained Model IG', 'Fully Randomized IG', '|Difference|']
for col_idx, title in enumerate(col_titles):
    axes[0, col_idx].set_title(title, fontsize=11, fontweight='bold')

for row_idx, model_name in enumerate(MODEL_NAMES):
    sub = full_rand_df[full_rand_df['model'] == model_name].sort_values('ssim')
    if sub.empty:
        for col_idx in range(3):
            axes[row_idx, col_idx].axis('off')
        continue

    # Pick the row with median SSIM (representative, not extreme outlier)
    rep = sub.iloc[len(sub) // 2]
    image_id = rep['image_id']
    key = (image_id, model_name)

    trained_map = baseline_maps.get(key)
    rand_p = RAND_PATH / f"{image_id}_{model_name}_full_rand.npy"
    if trained_map is None or not rand_p.exists():
        for col_idx in range(3):
            axes[row_idx, col_idx].axis('off')
        axes[row_idx, 0].set_ylabel(f"{model_name}\n(map not found)", fontsize=9)
        continue

    rand_map  = np.load(str(rand_p))
    diff_map  = np.abs(trained_map - rand_map)

    label = (f"{model_name}\n"
             f"SSIM={rep['ssim']:.3f}  "
             f"ρ={rep['spearman_r']:.3f}")
    axes[row_idx, 0].set_ylabel(label, fontsize=9)

    for col_idx, (arr, cmap, vmax) in enumerate([
        (trained_map, 'hot',    1.0),
        (rand_map,    'hot',    1.0),
        (diff_map,    'RdBu_r', float(diff_map.max()) or 1.0),
    ]):
        ax = axes[row_idx, col_idx]
        im = ax.imshow(arr, cmap=cmap, vmin=0, vmax=vmax)
        ax.axis('off')
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
panel_path = FIGPATH / 'figure_weight_rand_panel.png'
fig.savefig(str(panel_path), dpi=300, bbox_inches='tight')
plt.close(fig)
print(f"  ✅ {panel_path.name}")

# ── Figure 2: Cascading degradation curves (SSIM & Spearman) ───────────────

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(
    "Cascading Model Randomization — Attribution Map Degradation\n"
    "(Adebayo et al., 2018)  |  Lower = more degraded = better sanity check",
    fontsize=11, fontweight='bold',
)

palette = {
    'densenet121':    '#2196F3',
    'convnextv2_tiny': '#FF9800',
    'swinb_lora':     '#4CAF50',
}
markers = {
    'densenet121':    'o',
    'convnextv2_tiny': 's',
    'swinb_lora':     '^',
}
labels_nice = {
    'densenet121':    'DenseNet-121',
    'convnextv2_tiny': 'ConvNeXtV2-Tiny',
    'swinb_lora':     'SwinB-LoRA',
}

for ax, metric, ylabel in zip(
    axes,
    ['ssim', 'spearman_r'],
    ['SSIM (Structural Similarity)',
     'Spearman ρ (Rank Correlation)'],
):
    for mn in MODEL_NAMES:
        sub = cascade_df[cascade_df['model'] == mn].copy()
        if sub.empty:
            continue
        grouped = sub.groupby('pct_layers_rand')[metric]
        pcts  = sorted(grouped.groups.keys())
        means = [grouped.get_group(p).mean() for p in pcts]
        sems  = [grouped.get_group(p).sem()  for p in pcts]
        xs    = [p * 100 for p in pcts]

        ax.plot(xs, means,
                color=palette[mn], marker=markers[mn],
                label=labels_nice[mn], linewidth=2, markersize=8)
        ax.fill_between(
            xs,
            [m - s for m, s in zip(means, sems)],
            [m + s for m, s in zip(means, sems)],
            alpha=0.15, color=palette[mn],
        )

        # Dotted reference line at full randomization value
        sub_full = full_rand_df[full_rand_df['model'] == mn]
        if not sub_full.empty:
            ax.axhline(sub_full[metric].mean(),
                       color=palette[mn], linestyle=':', alpha=0.55,
                       linewidth=1.5)

    ax.set_xlabel('% of Layers Randomized (from classifier head → input)',
                  fontsize=10)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_xticks([25, 50, 75, 100])
    ax.set_xticklabels(['25%', '50%', '75%', '100%'])
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(15, 108)

plt.tight_layout()
cascade_fig_path = FIGPATH / 'figure_weight_rand_cascade.png'
fig.savefig(str(cascade_fig_path), dpi=300, bbox_inches='tight')
plt.close(fig)
print(f"  ✅ {cascade_fig_path.name}")


Generating figures...
  ✅ figure_weight_rand_panel.png
  ✅ figure_weight_rand_cascade.png


In [14]:
# ── Sanity assertions ──────────────────────────────────────────────────────

print("\n" + "=" * 65)
print("SANITY ASSERTIONS")
print("=" * 65)

checks = []

# 1. Full SSIM < 0.70 per model (randomized maps should be structurally different)
for mn in MODEL_NAMES:
    sub = full_rand_df[full_rand_df['model'] == mn]
    if sub.empty:
        checks.append((f"[{mn}] full SSIM < 0.70", False))
        continue
    val = sub['ssim'].mean()
    checks.append((f"[{mn}] full SSIM ({val:.3f}) < 0.70", val < 0.70))

# 2. Full Spearman < 0.50 (rank structure should degrade)
for mn in MODEL_NAMES:
    sub = full_rand_df[full_rand_df['model'] == mn]
    if sub.empty:
        checks.append((f"[{mn}] full Spearman < 0.50", False))
        continue
    val = sub['spearman_r'].mean()
    checks.append((f"[{mn}] full Spearman ρ ({val:.3f}) < 0.50", val < 0.50))

# 3. Full KLD > 0 (distribution should shift)
for mn in MODEL_NAMES:
    sub = full_rand_df[full_rand_df['model'] == mn]
    if sub.empty:
        checks.append((f"[{mn}] full KLD > 0", False))
        continue
    val = sub['kld'].mean()
    checks.append((f"[{mn}] full KLD ({val:.4f}) > 0", val > 0))

# 4. Cascading SSIM is non-increasing (25% → 100%)
for mn in MODEL_NAMES:
    sub = cascade_df[cascade_df['model'] == mn]
    if sub.empty:
        continue
    ssim_by_frac = {}
    for frac in CASCADE_FRACS:
        s = sub[np.isclose(sub['pct_layers_rand'], frac, atol=0.01)]['ssim']
        if not s.empty:
            ssim_by_frac[frac] = s.mean()
    if 0.25 in ssim_by_frac and 1.0 in ssim_by_frac:
        ok = ssim_by_frac[1.0] <= ssim_by_frac[0.25] + 0.05  # 0.05 tolerance
        checks.append((
            f"[{mn}] cascade SSIM non-increasing "
            f"(25%={ssim_by_frac[0.25]:.3f} → 100%={ssim_by_frac[1.0]:.3f})",
            ok,
        ))

# 5. Output files exist
for fname in [
    'weight_rand_full.csv',
    'weight_rand_cascade.csv',
    'weight_randomization_results.csv',
]:
    checks.append((f"{fname} exists", (RESULTS_PATH / fname).exists()))

for fname in [
    'figure_weight_rand_panel.png',
    'figure_weight_rand_cascade.png',
]:
    checks.append((f"{fname} exists", (FIGPATH / fname).exists()))

all_pass = True
for name, ok in checks:
    print(f"  [{'PASS' if ok else 'FAIL'}] {name}")
    if not ok:
        all_pass = False

if all_pass:
    print("\n✅ NB10 Weight Randomization Sanity Check — ALL CHECKS PASSED")
    print("   IG maps are sensitive to model parameters. Adebayo et al. test: PASS ✓")
else:
    failed = [n for n, ok in checks if not ok]
    print(f"\n⚠️  Checks failed: {failed}")
    print("   If Swin SSIM remains high: verify get_randomizable_layers includes backbone.")


SANITY ASSERTIONS
  [PASS] [densenet121] full SSIM (0.160) < 0.70
  [PASS] [convnextv2_tiny] full SSIM (0.077) < 0.70
  [PASS] [swinb_lora] full SSIM (0.256) < 0.70
  [FAIL] [densenet121] full Spearman ρ (0.512) < 0.50
  [PASS] [convnextv2_tiny] full Spearman ρ (0.263) < 0.50
  [PASS] [swinb_lora] full Spearman ρ (-0.164) < 0.50
  [PASS] [densenet121] full KLD (0.7427) > 0
  [PASS] [convnextv2_tiny] full KLD (1.4903) > 0
  [PASS] [swinb_lora] full KLD (0.7708) > 0
  [PASS] [densenet121] cascade SSIM non-increasing (25%=0.205 → 100%=0.171)
  [PASS] [convnextv2_tiny] cascade SSIM non-increasing (25%=0.479 → 100%=0.057)
  [PASS] [swinb_lora] cascade SSIM non-increasing (25%=0.458 → 100%=0.235)
  [PASS] weight_rand_full.csv exists
  [PASS] weight_rand_cascade.csv exists
  [PASS] weight_randomization_results.csv exists
  [PASS] figure_weight_rand_panel.png exists
  [PASS] figure_weight_rand_cascade.png exists

⚠️  Checks failed: ['[densenet121] full Spearman ρ (0.512) < 0.50']
   If Swin S

## Methods for Paper — Ablation Study: Weight Randomization Sanity Check

**Sanity check (Adebayo et al., 2018).** We applied cascading and full weight randomization to verify IG attributions depend on learned parameters. All layers including the frozen Swin backbone were eligible for re-initialization. Attribution maps were compared via SSIM, Spearman ρ, KL divergence, and entropy ratio.